In [ ]:
# This Should be the first running cell.

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/gamma4:latest'
CONFIG = f'{DRIVE_BASE}/config'
MODEL = f'{DRIVE_BASE}/models'
BACKUP = f'{DRIVE_BASE}/backup'

# Verification

for d in [CONFIG,MODEL,BACKUP]:
  os.makedirs(d,exist_ok=True)
  print(f'OK:{d}')


In [ ]:
# Extracting .env file credentials from specified drive.

from dotenv import load_dotenv
import subprocess

subprocess.run(['pip','install','python-dotenv'])
env_path = f'{CONFIG}/.env'

if not os.path.exists(env_path):
  raise FileNotFoundError(f'Upload .env at {env_path}')

load_dotenv(env_path) #Returns true if available

#Extracting API_keys from the .env file
WEBUI_SECRET_KEY = os.getenv('WEBUI_SECRET_KEY')
WEBUI_NAME = os.getenv('WEBUI_NAME')
WEBUI_URL = os.getenv('WEBUI_URL')
API_KEY = os.getenv('API_KEY')
CF_TOKEN = os.getenv('CLOUDFLARE_TUNNEL_TOKEN')

print('Environment Variables loaded: successfully')
print(f'WEBUI_Name: {WEBUI_NAME} , WEBUI_URL: {WEBUI_URL} (Click to access)')



In [ ]:
 # Need to manually install Ollam by :

!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Downloading the specific AI Model
!ollama run gamma4:latest

In [ ]:
import subprocess, sys
import urllib.request

#Function to run in colab
def run(cmd, **kargs):
  result = subprocess.run(cmd, shell = True, capture_output=True, text=True, **kargs)
  if result.returncode != 0:
    print(f'Error: {result.stderr}')
  return result

# Intall FASTAPI and uvicorn
print('Installing FASTAPI and uvicorn...')
run('pip install fastapi uvicorn httpx python-dotenv -q')

# Install Open WEBUI
print('Installing Open WebUI...')
run('pip install open-webui -q')

# Install cloudflared
print('Installing cloudflared...')
run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared')
run('chmod +x /usr/local/bin/cloudflared'
)

print('All dependencies installed')

In [ ]:
import subprocess, time, requests

# Ollama in background
ollama_pr = subprocess.Popen(
    ['ollama','serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Waiting for ollama
for i in range(30):
  try:
    r = requests.get('https://localhost:11434')
    if r.status_code == 200:
      print(f'Ollama ready after {1 + i} seconds')
      break
  except:
    time.sleep(1)
  else:
    raise RuntimeError(f'Ollama is unable to start after 30 seconds')


In [ ]:
import os, shutil, subprocess

OLLAMA_MODELS_DIR = os.path.expanduser('~/.ollama/models')
DRIVE_MODEL_CACHE = f'{MODEL}/ollama_cache'
MODEL_NAME = 'gemma4:latest'  #Can also use another model meeting colab resources requirements

def model_cached_locally():
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
    return MODEL_NAME in result.stdout

def model_on_drive():
    return os.path.exists(DRIVE_MODEL_CACHE) and os.listdir(DRIVE_MODEL_CACHE)

if model_cached_locally():
    print(f'Model already loaded: {MODEL_NAME}')
elif model_on_drive():
    print('Restoring model from Drive cache (30-60 sec)...')
    os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
    shutil.copytree(DRIVE_MODEL_CACHE, OLLAMA_MODELS_DIR, dirs_exist_ok=True)
    print('Model restored from Drive')

else:
    print('Pulling model from Ollama registry (~15 min on first run)...')
    result = subprocess.run(['ollama', 'pull', MODEL_NAME], capture_output=False)
    # Cache to Drive for future sessions
    print('Caching model to Drive for next session...')
    shutil.copytree(OLLAMA_MODELS_DIR, DRIVE_MODEL_CACHE, dirs_exist_ok=True)
    print('Model cached to Drive')

# Apply custom parameters from modelfile
modelfile_path = f'{CONFIG}/gamma4-params.modelfile'
if os.path.exists(modelfile_path):
    subprocess.run(['ollama', 'create', 'gemma4-custom', '-f', modelfile_path])
    print('Custom model parameters applied')



In [ ]:
import shutil, subprocess, os

# FastAPI from drive to notebook
shutil.copy(f'{CONFIG}/FastAPI-middleware.py', '/content/FastAPI-middleware.py')

# Environment for server process
env = os.environ.copy()
env['API_KEY'] = API_KEY

d = chr(45)
host_para = d + d + 'host'
port_para = d + d + 'post'

exec_cmd = ['uvicorn','api_server:app', host_para, '0.0.0.0', port_para, '8000' ]

# FastAPI on port 8000
api_process = subprocess.Popen(
    exec_cmd,
    env = env,
    stdout = subprocess.DEVNULL,
    stderr = subprocess.DEVNULL
)
import time; time.sleep(3)


In [ ]:
# Open WEBUI Setup

import subprocess, os, time, requests

webui_env = os.environ.copy()
webui_env['WEBUI_AUTH'] = 'false'
webui_env['ENABLE_SIGNUP'] = 'true'
webui_env['DEFAULT_USER_ROLE'] = 'admin'
webui_env.update({
    'OLLAMA_BASE_URL':  'http://localhost:11434',
    'WEBUI_SECRET_KEY': WEBUI_SECRET_KEY,
    'WEBUI_NAME':       WEBUI_NAME,
    'WEBUI_URL':        WEBUI_URL,
    'ENABLE_SIGNUP':    'false',                         #Setup admin account,default_models for specific use case
    'DEFAULT_MODELS':   'gemma4:latest',
    'DATA_DIR':         '/content/webui_data',

})

os.makedirs('/content/webui_data', exist_ok=True)

webui_proc = subprocess.Popen(
    ['open-webui', 'serve', '--port', '3000'],
    env=webui_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait up to 60 seconds for Open WebUI to be ready
print('Waiting for Open WebUI...')
for i in range(60):
    try:
        r = requests.get('http://localhost:3000')
        if r.status_code in [200, 302]:
            print(f'Open WebUI ready after {i+1}s')
            break
    except:
        time.sleep(1)
else:
    print('WARNING: Open WebUI may not be ready. Check logs.')


In [ ]:
 #Create CloudFlare Tunnel first
import subprocess

if CF_TOKEN:
  cf_process = subprocess.Popen(
      ['cloudflared','tunnel','--no-autoupdate','run','--token',CF_TOKEN],
      stdout = subprocess.DEVNULL,
      stderr = subprocess.DEVNULL
  )
  print(f'Cloudflared Tunnel started at {WEBUI_URL}')
else:
  print('Unable to obtain CF_TOKEN')

In [ ]:
# JS Integtion to keep server alive

from IPython.display import Javascript, display

keep_alive_js = '''
function keepAlive() {
  console.log('Keep-alive ping: ' + new Date().toLocaleTimeString());
  // Simulate a keypress to prevent idle detection
  document.dispatchEvent(new KeyboardEvent('keydown', {key: 'ArrowDown'}));
}
# Run every 60 seconds
if (window._keepAliveInterval) clearInterval(window._keepAliveInterval);
window._keepAliveInterval = setInterval(keepAlive, 60000);
console.log('Keep-alive started. Session will stay active.');
'''

display(Javascript(keep_alive_js))
print('Keep-alive active. Do not close this browser tab.')
print('Sessions can last up to 12 hours with this method.')


In [ ]:
import requests, json

print('=== Service Status ===')
print(f'Ollama PID:    {ollama_pr.pid}')
print(f'FastAPI PID:   {api_process.pid}')
print(f'Open WebUI PID:{webui_proc.pid}')
print()

In [ ]:
# Session Backup (need to run when closing colab)
from typing import Tuple

import shutil, datetime, os

timestamp = datetime.datetime.now().strftime('%d-%m-%Y__%H%M')
src = '/content/webui_data'
dest = f'{BACKUP}/webui_data_{timestamp}'
shutil.copytree(src,dest,dirs_exist_ok=True)

if os.path.exists(src):
  print(f'Chat backed up to {dest} successfully')
else:
  print('Chat not backed up or already updated')